In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import PIL
from PIL import Image

import os
from os.path import join, exists
from tqdm import tqdm

seed = 42
np.random.seed(seed)

In [2]:
BASE_DATA_PATH = '/home/orrin/aps360/pediatric-cxr-model/data_cleaned'
IMAGE_PATH = join(BASE_DATA_PATH, 'images')
ANNOT_PATH = join(BASE_DATA_PATH, 'annotations')

In [4]:
df = pd.read_csv(join(ANNOT_PATH, 'master_alternate_augmented.csv'))
print(df.value_counts('label'))
mask = df['label'] == 'no finding'
idx = np.where(mask)[0]


label
no finding           7932
pneumonia            4722
other                1477
heart disease        1088
consolidation        1088
infiltration         1088
effusion             1088
bronchiolitis        1088
bronchitis           1088
broncho-pnuemonia    1088
Name: count, dtype: int64


In [ ]:
# load images and labels
images = []
labels = []

for i in tqdm(range(len(df))):
    row = df.iloc[i]
    fname = row['filename']
    # image
    img_path = os.path.join(IMAGE_PATH, fname)
    img = np.array(Image.open(img_path).convert("L")).flatten()
    images.append(img)
    # label
    labels.append(row['label'])
images = np.array(images)

100%|██████████| 21747/21747 [01:04<00:00, 335.08it/s]


In [16]:
frac_train = 0.7
frac_val = 0.15
frac_test = 0.15

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

X_train, X_tmp, Y_train, Y_tmp = train_test_split(images, labels, test_size=frac_val+frac_test)
X_valid, X_test, Y_valid, Y_test = train_test_split(X_tmp, Y_tmp, test_size=frac_val/(frac_val+frac_test))

print(f"Number of Training: {len(Y_train)}")
print(f"Number of Validation: {len(Y_valid)}")
print(f"Number of Testing: {len(Y_test)}")

Number of Training: 700
Number of Validation: 150
Number of Testing: 150


In [17]:
# hyperparameters to try
hyp_n_estimators = [60, 80, 100, 120, 140]

val_acc = dict()

for n_est in tqdm(hyp_n_estimators):
    rfc = RandomForestClassifier(n_estimators=n_est, random_state=seed) 
    rfc.fit(X_train, Y_train)
    Y_valid_pred = rfc.predict(X_valid)
    acc = accuracy_score(Y_valid, Y_valid_pred)
    val_acc[n_est] = acc
    print(f"n_estimators: {n_est}, val_acc: {acc:.4f}")

 20%|██        | 1/5 [00:12<00:51, 12.90s/it]

n_estimators: 60, val_acc: 0.9067


 40%|████      | 2/5 [00:27<00:41, 13.69s/it]

n_estimators: 80, val_acc: 0.9000


 60%|██████    | 3/5 [00:44<00:30, 15.50s/it]

n_estimators: 100, val_acc: 0.9067


 80%|████████  | 4/5 [01:06<00:18, 18.13s/it]

n_estimators: 120, val_acc: 0.9067


100%|██████████| 5/5 [01:33<00:00, 18.72s/it]

n_estimators: 140, val_acc: 0.9067
